# Voting kết quả OCR của nhiều model

Gộp kết quả OCR của **ít nhất 3 model** cho cùng các trang (ví dụ PaddleOCR-VL, HunyuanOCR, API CLC của Lab) rồi bỏ
phiếu theo **từng chữ Hán**. Output của notebook này là **kết quả OCR cuối** của nhóm: dùng nó thay cho Output notebook
OCR ở mọi bước sau (dịch, `3_tao_input_test_chuoi`, `4_danh_gia`) và khi nộp bài.

**Add Input:** Output của các notebook OCR đã chạy (mỗi model một Output, có thư mục `ocr_raw/`). Nếu chạy OCR mà
không *Save Version* thì tải thư mục `ocr_raw/` về dạng zip, tạo Dataset từ file zip đó rồi *Add Input* cũng được. Thêm
`golden_<tên nhóm trưởng>` nếu muốn notebook in luôn CER của từng model và của bản voting. **Không cần GPU.**

**Cách voting của từng trang:**

1. Bỏ kết quả hỏng của model nào đó ở trang đó: trống, chạm trần token, hoặc lặp vòng.
2. Chọn **bản trục**: bản giống các bản còn lại nhất, tức tổng khoảng cách chỉnh sửa tới các bản khác nhỏ nhất. Hoà thì
   theo thứ tự `UU_TIEN`.
3. Gióng từng bản còn lại vào bản trục theo chữ Hán, bằng khoảng cách chỉnh sửa. So sánh không phân biệt giản thể với
   phồn thể.
4. Ở mỗi chữ của bản trục: chữ nào được nhiều model nhất thì giữ chữ đó; hoà thì giữ chữ của bản trục; nếu đa số model
   không có chữ này thì bỏ. Chữ mà bản trục thiếu chỉ được thêm vào khi đa số model cùng có.
5. Dấu câu và xuống dòng giữ theo bản trục.

Trang chỉ có 2 model thì không đủ đa số, lấy bản theo thứ tự `UU_TIEN`.

In [ ]:
UU_TIEN = ["hunyuan", "clc", "paddle"]  # thứ tự ưu tiên khi hoà, hoặc khi trang chỉ có 2 model

In [ ]:
!pip install -q rapidfuzz opencc-python-reimplemented

In [ ]:
# Dùng khi Input không có ocr_worker.py (ví dụ Input là zip Output tải về rồi tải lên lại)
OCR_WORKER_GON = r'''"""Bản rút gọn ocr_worker.py của notebook OCR mẫu: chỉ các hàm đọc kết quả OCR."""
from __future__ import annotations

import re


def md_to_lines(md: str) -> list[str]:
    """Markdown/HTML của OCR -> các dòng text thuần (bỏ #, **, ảnh, thẻ bảng, toạ độ hình)."""
    md = re.sub(r"</t[dh]>", " ", md)
    md = re.sub(r"</tr>|<br\s*/?>", "\n", md)
    md = re.sub(r"<[^>]+>", "", md)
    md = re.sub(r"!\[[^\]]*\]\([^)]*\)", "", md)
    out = []
    for ln in md.splitlines():
        ln = re.sub(r"^\s{0,3}#{1,6}\s*", "", ln)
        ln = re.sub(r"^\s*[-*+]\s+(?=\S)", "", ln)
        ln = re.sub(r"(?:[ \t,，]*\(\d{1,4},\d{1,4}\),\(\d{1,4},\d{1,4}\))+\s*$", "", ln)  # toạ độ hình của Hunyuan
        ln = ln.replace("**", "").replace("__", "").strip()
        if ln.startswith("|") and ln.endswith("|"):
            ln = " ".join(c.strip() for c in ln.strip("|").split("|") if c.strip())
        if ln and not re.fullmatch(r"[-|:\s]+", ln):
            out.append(ln)
    return out


def page_lines(rec: dict) -> list[str]:
    """1 trang OCR -> các dòng chữ theo thứ tự đọc."""
    if "blocks" not in rec:  # hunyuan: text markdown của cả trang
        return md_to_lines(rec["text"])
    ordered = sorted((b for b in rec["blocks"] if b["order"] is not None), key=lambda b: b["order"])
    center = lambda b: ((b["box"][0] + b["box"][2]) / 2, (b["box"][1] + b["box"][3]) / 2)  # noqa: E731
    for b in (b for b in rec["blocks"] if b["order"] is None):  # bảng, chú thích hình: paddle không xếp thứ tự đọc
        if not ordered:
            ordered.append(b)
            continue
        cx, cy = center(b)
        k = min(range(len(ordered)), key=lambda i: (center(ordered[i])[0] - cx) ** 2 + (center(ordered[i])[1] - cy) ** 2)
        ordered.insert(k + (cy > center(ordered[k])[1]), b)  # đặt cạnh vùng gần nhất
    return [ln for b in ordered for ln in md_to_lines(b["text"])]
'''

import glob, json, re, shutil, sys, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from opencc import OpenCC
from rapidfuzz.distance import Levenshtein

WORK = Path("/kaggle/working"); RAW = WORK / "ocr_raw"; TXT = WORK / "txt"
RAW.mkdir(parents=True, exist_ok=True); TXT.mkdir(exist_ok=True)
workers = sorted(glob.glob("/kaggle/input/**/ocr_worker.py", recursive=True))
if workers:
    shutil.copy(workers[0], WORK / "ocr_worker.py")  # để notebook 3, 4 đọc được Output này
else:  # Input là zip Output tải về, không kèm ocr_worker.py: dùng bản rút gọn, chỉ có hàm đọc kết quả
    (WORK / "ocr_worker.py").write_text(OCR_WORKER_GON, encoding="utf-8")
sys.path.insert(0, str(WORK)); import ocr_worker as ow

# 1. Gom ocr_raw/ của mọi Input về chung một chỗ (giữ nguyên file của từng model)
for f in sorted(glob.glob("/kaggle/input/**/ocr_raw/*/*_shard*.jsonl", recursive=True)):
    f = Path(f)
    if f.name.startswith("vote_") or "_layout_shard" in f.name:  # hunyuan_layout_*: vài trang vẽ layout, không phải OCR
        continue
    dst = RAW / f.parent.name / f.name
    k = 1
    while dst.exists():  # trùng tên (2 phiên cùng model): đổi tên, vẫn giữ dạng *_shard*.jsonl
        dst = RAW / f.parent.name / f.name.replace("_shard", f"_shard{k}x"); k += 1
    dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy(f, dst)

def loop_ratio(t):
    t = re.sub(r"\s+", " ", t)
    if len(t) < 600:
        return 0.0
    ch = [t[i:i + 30] for i in range(0, len(t) - 30, 10)]
    return 1 - len(set(ch)) / len(ch)

trang = defaultdict(lambda: defaultdict(dict))  # sách -> trang (từ 0) -> engine -> chữ
hong = defaultdict(lambda: defaultdict(set))     # sách -> trang -> engine bị bỏ vì hỏng
for f in sorted(RAW.glob("*/*_shard*.jsonl")):
    eng = re.sub(r"_retry$", "", f.name.split("_shard")[0])
    for line in open(f, encoding="utf-8"):
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            continue
        t = "\n".join(ow.page_lines(r)).strip()
        if not t or r.get("hit_max") or loop_ratio(t) > 0.3:
            hong[f.parent.name][r["page"]].add(eng)
            continue
        trang[f.parent.name][r["page"]][eng] = t

HAN = re.compile(r"[㐀-鿿豈-﫿\U00020000-\U0003134f]")
cc = OpenCC("t2s")
def khoa(c):  # so 2 chữ không phân biệt giản / phồn
    return cc.convert(unicodedata.normalize("NFKC", c))
def rank(e):
    return UU_TIEN.index(e) if e in UU_TIEN else len(UU_TIEN)

def vote(bans):
    """bans: {engine: chữ của trang} -> (chữ sau voting, bản trục, tỉ lệ vị trí mọi model đồng ý)."""
    if len(bans) == 1:
        e = next(iter(bans)); return bans[e], e, 1.0
    seq = {e: "".join(khoa(c) for c in t if HAN.match(c)) for e, t in bans.items()}
    if len(bans) == 2:  # không đủ đa số: lấy bản theo UU_TIEN; đồng ý = tỉ lệ chữ hai bản khớp nhau
        e = min(bans, key=rank); o = next(x for x in bans if x != e)
        khop = sum(i2 - i1 for tag, i1, i2, _, _ in Levenshtein.opcodes(seq[e], seq[o]) if tag == "equal")
        return bans[e], e, (khop / len(seq[e]) if seq[e] else 1.0)
    truc = min(bans, key=lambda e: (sum(Levenshtein.distance(seq[e], seq[o]) for o in bans if o != e), rank(e)))
    P, pk = bans[truc], seq[truc]
    n, m = len(pk), len(bans)
    phieu = [Counter({pk[i]: 1}) for i in range(n)]       # phiếu cho chữ ở từng vị trí của bản trục
    dang = [dict() for _ in range(n)]                      # khoá -> dạng chữ gốc
    bo = [0] * n                                           # số model không có chữ ở vị trí này
    chen = defaultdict(Counter); chen_goc = {}             # khe trước vị trí j -> chuỗi được chèn
    for e, t in bans.items():
        if e == truc:
            continue
        s = seq[e]; goc = [c for c in t if HAN.match(c)]
        for tag, i1, i2, j1, j2 in Levenshtein.opcodes(pk, s):
            if tag == "equal":
                for k in range(i2 - i1):
                    phieu[i1 + k][pk[i1 + k]] += 1
            elif tag == "replace":
                c = min(i2 - i1, j2 - j1)
                for k in range(c):
                    phieu[i1 + k][s[j1 + k]] += 1; dang[i1 + k].setdefault(s[j1 + k], goc[j1 + k])
                for k in range(c, i2 - i1):
                    bo[i1 + k] += 1
                if j2 - j1 > c:
                    w = s[j1 + c:j2]; chen[i1 + c][w] += 1; chen_goc.setdefault(w, "".join(goc[j1 + c:j2]))
            elif tag == "delete":
                for k in range(i1, i2):
                    bo[k] += 1
            elif tag == "insert":
                w = s[j1:j2]; chen[i1][w] += 1; chen_goc.setdefault(w, "".join(goc[j1:j2]))
    def them(j):
        if chen[j]:
            w, sl = chen[j].most_common(1)[0]
            if sl > m / 2:
                return chen_goc[w]
        return ""
    out, i, dong_y = [], 0, 0
    for ch in P:
        if not HAN.match(ch):
            out.append(ch); continue
        out.append(them(i))
        top = max(phieu[i].values())
        if bo[i] > sum(phieu[i].values()):
            pass  # đa số model không có chữ này -> bỏ
        elif phieu[i][pk[i]] == top:
            out.append(ch)
        else:
            w = max((k for k, v in phieu[i].items() if v == top), key=lambda k: k == pk[i])
            out.append(dang[i].get(w, w))
        dong_y += phieu[i][pk[i]] == m
        i += 1
    out.append(them(n))
    return "".join(out), truc, (dong_y / n if n else 1.0)

# 2. Voting từng trang, ghi vote_shard0.jsonl và txt/
tong = []
for book in sorted(trang):
    recs, chua_du = [], 0
    for p in sorted(trang[book]):
        bans = trang[book][p]
        text, truc, dy = vote(bans)
        chua_du += len(bans) < 3
        recs.append({"page": p, "text": text, "engines": sorted(bans), "truc": truc, "dong_y": dy,
                     "bo_vi_hong": sorted(hong[book][p])})
    with open(RAW / book / "vote_shard0.jsonl", "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in recs)
    (TXT / f"{book}.txt").write_text("\n\n".join(r["text"] for r in recs if r["text"]) + "\n", encoding="utf-8")
    engs = sorted({e for d in trang[book].values() for e in d})
    dys = [r["dong_y"] for r in recs if r["dong_y"] is not None]
    tong.append((book, len(recs), engs, chua_du, sum(dys) / len(dys) if dys else None))

print(f"{'sách':<50}{'trang':>6}  {'model':<26}{'<3 model':>9}{'đồng ý':>8}")
for book, n, engs, chua_du, dy in tong:
    print(f"{book:<50}{n:>6}  {','.join(engs):<26}{chua_du:>9}{(f'{dy:.0%}' if dy is not None else '-'):>8}")
    if len(engs) < 3 or "clc" not in engs:
        print("   !!! chưa đúng yêu cầu: phải voting ít nhất 3 model, trong đó có CLC")
print("\n'đồng ý' = tỉ lệ chữ mà mọi model đọc giống nhau. Output: ocr_raw/ (đủ các model + vote), txt/ từ bản voting.")

In [ ]:
# 3. (Tuỳ chọn) CER của từng model và của bản voting so với golden.jsonl, nếu đã Add Input golden
gj = glob.glob("/kaggle/input/**/golden.jsonl", recursive=True)
if not gj:
    print("Không có golden.jsonl trong Input -> bỏ qua phần so sánh")
else:
    gold = {r["id"]: r["zh"] for r in map(json.loads, open(gj[0], encoding="utf-8"))}
    def norm(s):
        s = cc.convert(unicodedata.normalize("NFKC", s))
        return "".join(c for c in s if unicodedata.category(c)[0] not in "PSZC")
    vot = {b: {r["page"]: r["text"] for r in map(json.loads, open(RAW / b / "vote_shard0.jsonl", encoding="utf-8"))}
           for b in trang}
    engs = sorted({e for b in trang for d in trang[b].values() for e in d}) + ["vote"]
    print(f"{'sách':<50}{'số trang':>9}" + "".join(f"{e:>10}" for e in engs))
    for b in sorted(trang):
        co = sorted({e for d in trang[b].values() for e in d})
        chung = [p for p in trang[b] if all(e in trang[b][p] for e in co)]  # chỉ so trên trang mọi model đều có
        E, N = Counter(), 0
        for p in chung:
            g = norm(gold.get(f"{b}__p{p + 1}", ""))
            if not g:
                continue
            N += len(g)
            for e in co + ["vote"]:
                t = vot[b].get(p, "") if e == "vote" else trang[b][p][e]
                E[e] += min(Levenshtein.distance(norm(t), g), len(g))
        if N:
            print(f"{b:<50}{len(chung):>9}" + "".join(f"{E[e] / N:10.3f}" if e in co + ['vote'] else f"{'-':>10}" for e in engs))
    print("Chỉ so trên các trang mà mọi model của sách đó đều có kết quả, để không model nào bị thiệt vì thiếu trang.")
    print("Lưu ý: chữ Hán tham chiếu được đọc bằng HunyuanOCR nên HunyuanOCR có lợi thế khi so; chỗ voting khác tham chiếu")
    print("cần mở ảnh trang xem bên nào đúng, dùng file kiem_tay.csv ở cell dưới.")

In [ ]:
# 4. Lấy ngẫu nhiên SO_CHO chỗ bản voting khác chữ tham chiếu -> kiem_tay.csv để nhóm mở ảnh trang kiểm tay
SO_CHO = 30
import csv, random
if gj:
    def han(t):
        return [c for c in t if HAN.match(c)]
    def doi_vi_tri(a, b):  # ranh giới k trong a -> khoảng [lo, hi] trong b (gồm cả chữ b chèn thêm ở đó)
        lo, hi = [len(b)] * (len(a) + 1), [0] * (len(a) + 1)
        for tag, i1, i2, j1, j2 in Levenshtein.opcodes(a, b):
            for k in range(i1, i2 + 1):
                if tag == "insert":
                    x, y = j1, j2
                else:
                    x = y = j1 if tag == "delete" else (j2 if k == i2 else min(j1 + k - i1, j2))
                lo[k], hi[k] = min(lo[k], x), max(hi[k], y)
        return lo, hi
    cho = []
    for b in sorted(trang):
        for p, text in vot[b].items():
            g = han(gold.get(f"{b}__p{p + 1}", ""))
            v = han(text)
            if not g or not v:
                continue
            vk, gk = "".join(map(khoa, v)), "".join(map(khoa, g))
            ban = {e: han(t) for e, t in trang[b][p].items()}
            maps = {e: (doi_vi_tri(vk, "".join(map(khoa, s))), s) for e, s in ban.items()}
            for tag, i1, i2, j1, j2 in Levenshtein.opcodes(vk, gk):
                if tag == "equal" or max(i2 - i1, j2 - j1) > 4:
                    continue  # bỏ chỗ lệch dài, thường do đọc khác thứ tự
                row = {"id": f"{b}__p{p + 1}",
                       "ngu_canh": "".join(v[max(0, i1 - 6):i1]) + "【" + "".join(v[i1:i2]) + "】" + "".join(v[i2:i2 + 6]),
                       "vote": "".join(v[i1:i2]) or "(không có)", "tham_chieu": "".join(g[j1:j2]) or "(không có)"}
                for e, ((lo, hi), s) in maps.items():
                    row[e] = "".join(s[lo[i1]:hi[i2]]) or "(không có)"
                cho.append(row)
    random.seed(0)
    mau = random.sample(cho, min(SO_CHO, len(cho)))
    engs_all = sorted({e for b in trang for d in trang[b].values() for e in d})
    with open(WORK / "kiem_tay.csv", "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, ["id", "ngu_canh", "vote", "tham_chieu"] + engs_all + ["ben_dung", "ghi_chu"], restval="")
        w.writeheader(); w.writerows(sorted(mau, key=lambda r: r["id"]))
    print(f"Có {len(cho)} chỗ bản voting khác chữ tham chiếu, đã lấy ngẫu nhiên {len(mau)} chỗ vào kiem_tay.csv.")
    print("Mở ảnh từng trang, tìm đoạn ở cột ngu_canh, điền cột ben_dung: vote, tham_chieu, ca_hai_sai hoặc khong_ro.")